# Pretrained

Celem eksperymentu jest porównanie dwóch metod reprezentacji tekstu: TF-IDF oraz embeddingów z modelu `sentence-transformers/all-MiniLM-L6-v2`.

W obu podejściach wykorzystano Logistic Regression. Wyniki porównano na tym samym zbiorze validation, używając macro-F1 jako głównej metryki.

## 1. Model pretrained

Wykorzystany został model sentence-transformers/all-MiniLM-L6-v2, który zamienia tekst na wektory liczbowe (embeddingi).

Model opiera się na encoderze Transformer z 6 warstwami. Był uczony na ponad miliardzie par tekstów z różnych źródeł, m.in. Reddit i S2ORC. Do uczenia reprezentacji zdań wykorzystano metodę kontrastową, której celem jest zbliżanie reprezentacji powiązanych ze sobą tekstów.

Każda nazwa produktu zostaje zamieniona na wektor 384 liczb. Model wykorzystuje mean pooling, czyli uśrednianie reprezentacji tokenów. Otrzymane wektory są następnie normalizowane.

Parametry MiniLM pozostają zamrożone. Trenowana jest wyłącznie Logistic Regression na danych train.

Model jest udostępniany na licencji Apache 2.0. Limit wejścia wynosi 256 tokenów, a dłuższe teksty są ucinane.

Źródło: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

## 2. Dane i środowisko

In [4]:
from pathlib import Path
import json
import platform
import resource
import sys
import time

import numpy as np
import pandas as pd
import sklearn
import torch
import sentence_transformers
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

SEED = 42
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
DEVICE = "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd()
if not (ROOT / "splits" / "train_ids.csv").is_file():
    ROOT = ROOT.parent

DATA_NAME = "product+classification+and+clustering.zip"
DATA_PATHS = [ROOT / "data" / DATA_NAME, ROOT / "data" / "sample" / DATA_NAME]
DATA_PATH = next((p for p in DATA_PATHS if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Umieść ZIP z UCI lokalnie w data/ lub data/sample/.")

RESULTS = ROOT / "results" / "validation"
RESULTS.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__, "pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__, "sentence-transformers:", sentence_transformers.__version__)
print("Sprzęt:", DEVICE, platform.system(), platform.machine())

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 19:47:18.415000 5534 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Python: 3.14.2
numpy: 2.4.0 pandas: 2.3.3 scikit-learn: 1.8.0
torch: 2.15.0 sentence-transformers: 6.3.0
Sprzęt: cpu Darwin arm64


In [5]:
df = pd.read_csv(DATA_PATH, compression="zip")
df.columns = df.columns.str.strip()
assert df["Product ID"].is_unique

train_ids = pd.read_csv(ROOT / "splits" / "train_ids.csv")
val_ids = pd.read_csv(ROOT / "splits" / "validation_ids.csv")

# Kolejność identyfikatorów jest ustalona przez pliki splitów.
by_id = df.set_index("Product ID")
train = by_id.loc[train_ids["id"].tolist()].reset_index()
val = by_id.loc[val_ids["id"].tolist()].reset_index()

assert set(train["Cluster ID"]).isdisjoint(set(val["Cluster ID"]))
normalized = lambda s: s.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
assert set(normalized(train["Product Title"])).isdisjoint(set(normalized(val["Product Title"])))

X_train = train["Product Title"].astype(str).tolist()
y_train = train["Category Label"].astype(str)
X_val = val["Product Title"].astype(str).tolist()
y_val = val["Category Label"].astype(str)

print("Train:", len(train), "ofert /", train["Cluster ID"].nunique(), "produktów")
print("Validation:", len(val), "ofert /", val["Cluster ID"].nunique(), "produktów")
print("Liczba klas:", y_train.nunique())

Train: 28338 ofert / 10684 produktów
Validation: 3479 ofert / 1282 produktów
Liczba klas: 10


## 3. Zamrożony encoder i embeddingi

In [6]:
t = time.perf_counter()
encoder = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device=DEVICE)
model_load_seconds = time.perf_counter() - t

encoder.eval()
for param in encoder.parameters():
    param.requires_grad_(False)

assert not any(p.requires_grad for p in encoder.parameters())
print("Rozmiar embeddingu:", encoder.get_sentence_embedding_dimension())
print("Limit tokenów:", encoder.max_seq_length)
print("Czas wczytania modelu (s):", round(model_load_seconds, 2))

# Sprawdzenie limitu wyłącznie dla train i validation.
lengths_train = [len(ids) for ids in encoder.tokenizer(X_train, truncation=False, padding=False)["input_ids"]]
lengths_val = [len(ids) for ids in encoder.tokenizer(X_val, truncation=False, padding=False)["input_ids"]]
truncated_train = sum(n > encoder.max_seq_length for n in lengths_train)
truncated_val = sum(n > encoder.max_seq_length for n in lengths_val)
print("Tytuły powyżej limitu — train:", truncated_train, "/", len(X_train))
print("Tytuły powyżej limitu — validation:", truncated_val, "/", len(X_val))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25115.59it/s]
/var/folders/l6/v1_y376j7wj_zzzvznk1jc6c0000gn/T/ipykernel_5534/941343052.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Rozmiar embeddingu:", encoder.get_sentence_embedding_dimension())


Rozmiar embeddingu: 384
Limit tokenów: 256
Czas wczytania modelu (s): 19.89
Tytuły powyżej limitu — train: 0 / 28338
Tytuły powyżej limitu — validation: 0 / 3479


In [7]:
t = time.perf_counter()
emb_train = encoder.encode(
    X_train, batch_size=32, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
embedding_train_seconds = time.perf_counter() - t

t = time.perf_counter()
emb_val = encoder.encode(
    X_val, batch_size=32, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
embedding_val_seconds = time.perf_counter() - t

assert emb_train.shape == (len(train), 384)
assert emb_val.shape == (len(val), 384)
print("Embeddingi train:", emb_train.shape, "czas (s):", round(embedding_train_seconds, 2))
print("Embeddingi validation:", emb_val.shape, "czas (s):", round(embedding_val_seconds, 2))

Batches: 100%|██████████| 109/109 [00:02<00:00, 49.46it/s]

Embeddingi train: (28338, 384) czas (s): 18.35
Embeddingi validation: (3479, 384) czas (s): 2.21


## 4. Logistic Regression i wyniki na validation

In [8]:
classifier = LogisticRegression(max_iter=1000, random_state=SEED)
t = time.perf_counter()
classifier.fit(emb_train, y_train)
classifier_fit_seconds = time.perf_counter() - t

t = time.perf_counter()
pred_pretrained = classifier.predict(emb_val)
prediction_seconds = time.perf_counter() - t

def metrics(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

pretrained_metrics = metrics(y_val, pred_pretrained)
with open(RESULTS / "tfidf_lr_metrics.json", encoding="utf-8") as f:
    baseline_info = json.load(f)
baseline_metrics = baseline_info["metrics"]

comparison = pd.DataFrame([
    {"model": "TF-IDF + Logistic Regression", **baseline_metrics},
    {"model": "MiniLM + Logistic Regression", **pretrained_metrics},
]).set_index("model")
print(comparison.round(4).to_string())
print("Różnica macro-F1 (MiniLM - TF-IDF):", round(pretrained_metrics["macro_f1"] - baseline_metrics["macro_f1"], 4))
print("\nF1 dla poszczególnych klas — MiniLM:")
print(classification_report(y_val, pred_pretrained, zero_division=0))

                              macro_f1  accuracy  weighted_f1
model                                                        
TF-IDF + Logistic Regression    0.9627    0.9623       0.9623
MiniLM + Logistic Regression    0.9695    0.9695       0.9695
Różnica macro-F1 (MiniLM - TF-IDF): 0.0068

F1 dla poszczególnych klas — MiniLM:
                  precision    recall  f1-score   support

            CPUs       1.00      1.00      1.00       387
 Digital Cameras       1.00      0.98      0.99       270
     Dishwashers       0.98      0.95      0.97       341
        Freezers       0.96      0.94      0.95       222
 Fridge Freezers       0.94      0.97      0.96       550
         Fridges       0.94      0.92      0.93       359
      Microwaves       0.96      0.97      0.97       235
   Mobile Phones       0.97      0.98      0.98       408
             TVs       0.99      0.99      0.99       356
Washing Machines       0.97      0.97      0.97       351

        accuracy               

## 5. Gdzie MiniLM pomaga, a gdzie nie

In [9]:
baseline_pred_df = pd.read_csv(RESULTS / "tfidf_lr_predictions.csv")
assert baseline_pred_df["id"].is_unique
baseline_aligned = baseline_pred_df.set_index("id").loc[val["Product ID"].tolist()]
assert np.array_equal(baseline_aligned["true_label"].astype(str).to_numpy(), y_val.to_numpy())
pred_tfidf = baseline_aligned["predicted_label"].astype(str).to_numpy()

cases = pd.DataFrame({
    "id": val["Product ID"].to_numpy(),
    "title": X_val,
    "true_label": y_val.to_numpy(),
    "tfidf_pred": pred_tfidf,
    "minilm_pred": pred_pretrained,
})

case_groups = [
    ("MiniLM poprawny, TF-IDF błędny", (cases.tfidf_pred != cases.true_label) & (cases.minilm_pred == cases.true_label)),
    ("TF-IDF poprawny, MiniLM błędny", (cases.tfidf_pred == cases.true_label) & (cases.minilm_pred != cases.true_label)),
    ("Oba modele błędne", (cases.tfidf_pred != cases.true_label) & (cases.minilm_pred != cases.true_label)),
]
examples = []
for label, mask in case_groups:
    subset = cases.loc[mask].head(3).copy()
    subset.insert(0, "przypadek", label)
    examples.append(subset)
    print("\n", label, ":", int(mask.sum()), "przykładów")
    if not subset.empty:
        print(subset[["title", "true_label", "tfidf_pred", "minilm_pred"]].to_string(index=False))

examples_df = pd.concat(examples, ignore_index=True)
examples_df.to_csv(RESULTS / "minilm_lr_comparison_examples.csv", index=False)


 MiniLM poprawny, TF-IDF błędny : 65 przykładów
                                              title      true_label      tfidf_pred     minilm_pred
                                        x100f black Digital Cameras   Mobile Phones Digital Cameras
kidizoom selfie cam ideal gift for all celebrations Digital Cameras   Mobile Phones Digital Cameras
                  panasonic nne27jwmbpq/nne27jwmbpq      Microwaves Digital Cameras      Microwaves

 TF-IDF poprawny, MiniLM błędny : 40 przykładów
                                   title    true_label    tfidf_pred minilm_pred
alcatel 5099d 2balwe2 15 24 cm 6 zoll 3v Mobile Phones Mobile Phones         TVs
   cubot r11 5 5 zoll hd 18 9 verh ltnis Mobile Phones Mobile Phones     Fridges
                       swisstone bbm 625 Mobile Phones Mobile Phones  Microwaves

 Oba modele błędne : 66 przykładów
                                                title true_label    tfidf_pred   minilm_pred
                        xoro xor400494 htc 1946 b

## 6. Koszt i zapis wyników

In [10]:
# Maksymalne zużycie RAM przez proces notebooka (łącznie z bibliotekami).
max_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_ram_mb = max_rss / (1024 ** 2) if sys.platform == "darwin" else max_rss / 1024

times = {
    "model_load_seconds": round(model_load_seconds, 3),
    "embedding_train_seconds": round(embedding_train_seconds, 3),
    "embedding_validation_seconds": round(embedding_val_seconds, 3),
    "classifier_fit_seconds": round(classifier_fit_seconds, 3),
    "prediction_seconds": round(prediction_seconds, 3),
}
print("Czasy (s):", times)
print("Szczyt RAM procesu (MB):", round(peak_ram_mb, 1))
print("Baseline TF-IDF — czas fit (s):", baseline_info["training_seconds"])
print("Uwaga: wczytanie MiniLM może obejmować pierwsze pobranie wag.")

pd.DataFrame({
    "id": val["Product ID"].to_numpy(),
    "true_label": y_val.to_numpy(),
    "predicted_label": pred_pretrained,
}).to_csv(RESULTS / "minilm_lr_predictions.csv", index=False)

report = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "split": "validation",
    "seed": SEED,
    "metrics": pretrained_metrics,
    "times": times,
    "peak_process_ram_mb": round(peak_ram_mb, 1),
    "hardware": f"{DEVICE}, {platform.system()} {platform.machine()}",
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "scikit_learn": sklearn.__version__,
    "embedding_dim": int(encoder.get_sentence_embedding_dimension()),
    "max_seq_length": int(encoder.max_seq_length),
    "titles_over_limit_train": truncated_train,
    "titles_over_limit_validation": truncated_val,
}
(RESULTS / "minilm_lr_metrics.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Zapisano wyniki do: results/validation/")

Czasy (s): {'model_load_seconds': 19.887, 'embedding_train_seconds': 18.353, 'embedding_validation_seconds': 2.207, 'classifier_fit_seconds': 0.474, 'prediction_seconds': 0.001}
Szczyt RAM procesu (MB): 883.2
Baseline TF-IDF — czas fit (s): 1.1664
Uwaga: wczytanie MiniLM może obejmować pierwsze pobranie wag.
Zapisano wyniki do: results/validation/


/var/folders/l6/v1_y376j7wj_zzzvznk1jc6c0000gn/T/ipykernel_5534/845227813.py:36: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "embedding_dim": int(encoder.get_sentence_embedding_dimension()),


## 7. Krótka interpretacja

MiniLM + Logistic Regression uzyskał macro-F1 = 0.9695, czyli nieco wyższy wynik niż TF-IDF + Logistic Regression (0.9627).

MiniLM poprawnie sklasyfikował 65 przykładów, w których TF-IDF się pomylił, ale popełnił też 40 błędów w przykładach poprawnie rozpoznanych przez TF-IDF. Oba modele pomyliły się w 66 przypadkach.

Obliczenie embeddingów MiniLM trwało 18.35 s dla train i 2.21 s dla validation, a trenowanie Logistic Regression 0.47 s. Dla porównania trening pipeline'u TF-IDF trwał 1.17 s. MiniLM uzyskał więc nieco lepszy wynik, ale wymagał więcej czasu obliczeniowego.